# 3.10 AutoML: una búsqueda vale lo que vale su score

🖥️ [**Diapositivas de la clase: sesión 18 (lun 9 nov)**](https://geo-smart.github.io/mlgeo-book/slides/2026/lec18_robust_training.html)

Las herramientas de aprendizaje automático automatizado (AutoML), como H2O AutoML, AutoGluon y auto-sklearn, automatizan la parte central de un proyecto: probar muchas familias de modelos y configuraciones, calificar cada una, quedarse con la mejor y, a menudo, apilar las ganadoras. Por debajo hay un bucle lo bastante corto como para que usted lo escriba, así que este notebook de Jupyter (cuaderno) solo necesita scikit-learn. Las ediciones anteriores de este capítulo enseñaban pycaret y Optuna. Escribir el bucle a mano muestra lo que esos paquetes no pueden hacer por usted: ordenan los modelos según el score que usted les dé.

Lee et al. (2023, *Remote Sensing*) muestran por qué eso importa. AutoML superó a un bosque aleatorio para la humedad diaria del suelo en Corea del Sur con muestreo aleatorio (error de 2.21 % frente a 2.71 %). Cuando se dejaron fuera años completos, los dos empataron (4.13 % frente a 4.10 %).

**Resultados de aprendizaje.** Al terminar este notebook, usted podrá:

1. escribir un pequeño bucle de AutoML con scikit-learn;
2. mostrar que la elección de la validación puede cambiar al ganador, y medir lo que eso cuesta en una prueba final;
3. decir por qué el mejor score de una búsqueda es optimista;
4. auditar un script de modelado escrito por un agente de IA, y repararlo.

**Tiempo:** unos 20 minutos en parejas. Si le falta tiempo, omita la Parte 2.

## Cómo funciona este notebook

Cada celda **Su turno** pide unas pocas líneas de código. Lea la pista, escriba su respuesta donde lo indica la celda y ejecútela. La celda de abajo contiene una solución desarrollada, oculta hasta que haga clic en ⋯. Esa celda oculta también comprueba su respuesta: imprime `correct` o una pista, y si su respuesta falta o es distinta, usa el resultado de referencia para que el resto del notebook siga funcionando. Su propia respuesta, si es distinta, se conserva con el mismo nombre seguido de `_yours`.

In [1]:
# Data handling
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pooch

# Plotting
import matplotlib.pyplot as plt

# Models, splits and scores (scikit-learn)
from sklearn.datasets import make_classification   # synthetic stand-in if Zenodo cannot be reached
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Los valores siguientes se usan en varios lugares, así que se definen una sola vez. Los colores de las figuras se distinguen también para personas con daltonismo (la paleta Okabe-Ito).

In [2]:
SEED = 42
SPLIT_SEED = 2026   # the canonical split of notebook 3.5
BLUE, ORANGE, RED, MUTED = "#0072B2", "#E69F00", "#D55E00", "#6e675c"
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})

In [3]:
# Helper: compare an answer with a reference. If the answer is missing or different, print a hint and use the
# reference so the cells below still run; a different answer is kept as <name>_yours.
def check(name, ns, reference, hint, measure=None, tol=1e-3):
    measure = measure or (lambda value: np.asarray(value, dtype=float))
    yours = ns.get(name)
    if yours is None or (hasattr(yours, "__len__") and len(yours) == 0):
        print(f"{name}: not answered yet. Hint: {hint}\n  Using the reference result below.")
    else:
        try:
            ok = np.allclose(measure(yours), measure(reference), atol=tol)
        except Exception:
            ok = False
        if ok:
            print(f"{name}: correct")
            return
        print(f"{name}: not the expected result. Hint: {hint}\n  Your answer is kept as {name}_yours; "
              "the reference is used below.")
        ns[f"{name}_yours"] = yours
    ns[name] = reference

## Parte 1. Un pequeño AutoML, a mano

*Datos:* 📍 Sensor in situ · 🪨 Tierra sólida · 📐 Geodesia

Reutilizamos el pronóstico de [3.8](3.8_robust_training.ipynb): la estación GNSS P395 en la cordillera costera de Oregón, con 60 días de posición hacia el este para predecir la media de los 30 días siguientes. La búsqueda ve el primer 80 % del registro. El último 20 % queda fuera para una única prueba final.

In [4]:
course_path = Path("../Chapter1-GettingStarted/data/gps_P395_relative_position.csv")
if not course_path.exists():
    course_path = Path("gps_P395_relative_position.csv")
    if not course_path.exists():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/geo-smart/mlgeo-book/main/book/Chapter1-GettingStarted/data/gps_P395_relative_position.csv", course_path)
if hashlib.sha256(course_path.read_bytes()).hexdigest() != "eb22473bbc5565caafbfe1b3b539c65cf88ba3cb23dec07106d81738957a4ff6":
    raise ValueError(f"{course_path} is not the course copy. Delete it, or restore it with git, and run this cell again.")
gnss = pd.read_csv(course_path)
east = gnss["east_m"].to_numpy() * 1000   # mm, relative to the first day
year = gnss["decimal_year"].to_numpy()
history, horizon, stride = 60, 30, 5
rows = []
for i in range(history, len(east) - horizon, stride):
    past = east[i - history:i]
    rows.append({"mean_60d": past.mean(), "mean_10d": east[i - 10:i].mean(), "last": east[i - 1],
                 "slope_60d": np.polyfit(np.arange(history), past, 1)[0],
                 "season_sin": np.sin(2 * np.pi * year[i]), "season_cos": np.cos(2 * np.pi * year[i]),
                 "target": east[i:i + horizon].mean(), "year": year[i]})
samples = pd.DataFrame(rows)
FEATURES = ["mean_60d", "mean_10d", "last", "slope_60d", "season_sin", "season_cos"]
X_time, y_time = samples[FEATURES].to_numpy(), samples["target"].to_numpy()
last_value = samples["last"].to_numpy()
GAP = horizon // stride   # samples left out between training and validation, so target windows do not overlap
print(f"{len(samples)} samples, {X_time.shape[1]} features")

cut = int(0.8 * len(X_time))
X_search, y_search = X_time[:cut], y_time[:cut]
X_final, y_final = X_time[cut + GAP:], y_time[cut + GAP:]
print(f"{len(X_search)} samples to search on, {len(X_final)} for the final test")

1482 samples, 6 features
1185 samples to search on, 291 for the final test


Estos son los candidatos: familias de modelos y configuraciones que probaría una herramienta de AutoML.

In [5]:
candidates = {"Linear": LinearRegression(),
              "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
              "Nearest 3": make_pipeline(StandardScaler(), KNeighborsRegressor(3)),
              "Nearest 30": make_pipeline(StandardScaler(), KNeighborsRegressor(30)),
              "Forest": RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
              "Forest, leaf 20": RandomForestRegressor(n_estimators=100, min_samples_leaf=20, random_state=SEED, n_jobs=-1),
              "Boosted": HistGradientBoostingRegressor(random_state=SEED),
              "Boosted, slow": HistGradientBoostingRegressor(learning_rate=0.03, random_state=SEED)}
shuffled = KFold(n_splits=5, shuffle=True, random_state=SEED)
future = TimeSeriesSplit(n_splits=5, gap=GAP)

### Su turno (1): el bucle

Escriba `mini_automl(candidates, X, y, cv)`. Para cada candidato, calcule el error absoluto medio con validación cruzada y luego devuelva una Series de pandas con los errores ordenados del mejor al peor.

*Pista:*
```python
scores = {}
for name, model in candidates.items():
    scores[name] = -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()
return pd.Series(scores).sort_values()
```

In [6]:
mini_automl = None   # replace this line with your function: def mini_automl(candidates, X, y, cv): ...

*Solución desarrollada abajo (haga clic en ⋯ para abrirla).*

In [7]:
def mini_automl_sol(candidates, X, y, cv):
    scores = {}
    for name, model in candidates.items():
        scores[name] = -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()
    return pd.Series(scores).sort_values()


check("mini_automl", globals(), mini_automl_sol, "loop over candidates.items(), score each with cross_val_score, return a sorted Series.",
      measure=lambda f: f({"Linear": LinearRegression(), "Ridge": candidates["Ridge"]}, X_search, y_search, shuffled).to_numpy())

mini_automl: not answered yet. Hint: loop over candidates.items(), score each with cross_val_score, return a sorted Series.
  Using the reference result below.


### Su turno (2): ejecutar la búsqueda dos veces

Complete `ranking_shuffled` con su función sobre `X_search, y_search` con `cv=shuffled`, y `ranking_future` con `cv=future`. Antes de ejecutarla, anote qué modelo espera que gane en cada caso. Tarda unos 20 segundos.

In [8]:
ranking_shuffled = None   # replace None with your expression
ranking_future = None     # replace None with your expression

*Solución desarrollada abajo (haga clic en ⋯ para abrirla).*

In [9]:
ranking_shuffled_sol = mini_automl(candidates, X_search, y_search, shuffled)
ranking_future_sol = mini_automl(candidates, X_search, y_search, future)
check("ranking_shuffled", globals(), ranking_shuffled_sol, "mini_automl(candidates, X_search, y_search, shuffled)",
      measure=lambda s: s.sort_index().to_numpy(), tol=0.01)
check("ranking_future", globals(), ranking_future_sol, "mini_automl(candidates, X_search, y_search, future)",
      measure=lambda s: s.sort_index().to_numpy(), tol=0.01)
pd.DataFrame({"shuffled folds (mm)": ranking_shuffled, "future folds (mm)": ranking_future}).sort_values("shuffled folds (mm)").round(2)

ranking_shuffled: not answered yet. Hint: mini_automl(candidates, X_search, y_search, shuffled)
  Using the reference result below.
ranking_future: not answered yet. Hint: mini_automl(candidates, X_search, y_search, future)
  Using the reference result below.


,shuffled folds (mm),future folds (mm)
Forest,0.46,9.58
Boosted,0.58,9.78
Linear,0.77,0.82
Ridge,0.78,0.86
"Forest, leaf 20",0.97,10.40
Nearest 3,1.17,11.30
"Boosted, slow",1.46,10.94
Nearest 30,2.44,13.04


### Su turno (3): la prueba final, una sola vez

Cada búsqueda nombra a un ganador. Ajuste cada ganador sobre `X_search, y_search` y calcule su error absoluto medio sobre `X_final, y_final`: `final_shuffled` para el ganador con pliegues mezclados, `final_future` para el ganador con pliegues futuros.

*Pista:* `ranking_shuffled.index[0]` es el nombre del ganador y `candidates[name]` es el modelo. Luego, `mean_absolute_error(y_final, model.fit(X_search, y_search).predict(X_final))`.

In [10]:
final_shuffled = None   # replace None with your expression
final_future = None     # replace None with your expression

*Solución desarrollada abajo (haga clic en ⋯ para abrirla).*

In [11]:
final_shuffled_sol = mean_absolute_error(y_final, candidates[ranking_shuffled.index[0]].fit(X_search, y_search).predict(X_final))
final_future_sol = mean_absolute_error(y_final, candidates[ranking_future.index[0]].fit(X_search, y_search).predict(X_final))
check("final_shuffled", globals(), final_shuffled_sol, "take the first name in ranking_shuffled, fit that candidate, score it on the final samples.", tol=0.01)
check("final_future", globals(), final_future_sol, "take the first name in ranking_future, fit that candidate, score it on the final samples.", tol=0.01)
print(f"Winner on shuffled folds: {ranking_shuffled.index[0]}, final test {final_shuffled:.2f} mm")
print(f"Winner on future folds:   {ranking_future.index[0]}, final test {final_future:.2f} mm")
print(f"Persistence, final test:  {mean_absolute_error(y_final, X_final[:, FEATURES.index('last')]):.2f} mm")

final_shuffled: not answered yet. Hint: take the first name in ranking_shuffled, fit that candidate, score it on the final samples.
  Using the reference result below.
final_future: not answered yet. Hint: take the first name in ranking_future, fit that candidate, score it on the final samples.
  Using the reference result below.
Winner on shuffled folds: Forest, final test 13.36 mm
Winner on future folds:   Linear, final test 0.74 mm
Persistence, final test:  1.12 mm


**Haga una pausa y discuta.**

- El mismo bucle, los mismos candidatos y los mismos datos eligieron dos modelos distintos. ¿Qué cambió en el bucle?
- Lee et al. (2023) encontraron a AutoML por delante con muestreo aleatorio y empatado cuando se dejaron fuera años completos. ¿Qué comparación corresponde a la humedad diaria del suelo del año próximo?
- ¿Qué le preguntaría a un artículo que reporta «el mejor de 20 modelos seleccionados por AutoML»?

:::{dropdown} Puntos a destacar
Solo cambió la validación. Una búsqueda premia lo que premie el score, y los pliegues mezclados premian copiar a los vecinos, algo que los bosques hacen bien. Luego, pregúntele al artículo: qué se dejó fuera, qué modelo de referencia (*baseline*) superó el ganador, y si el score reportado se usó para elegir al ganador. Los paquetes ahorran tiempo en el bucle. No saben cómo se usará su modelo.
:::

## Parte 2. Buscar configuraciones, y por qué el mejor score es optimista

El bucle se extiende a las configuraciones: extraer un modelo al azar, calificarlo, quedarse con el mejor. `GridSearchCV` y `RandomizedSearchCV` de scikit-learn hacen lo mismo con menos código. Bibliotecas como Optuna extraen nuevas configuraciones cerca de las que obtuvieron buenos scores, lo que ahorra tiempo cuando cada ajuste tarda horas, y optimizan el mismo score.

Cuantos más candidatos prueba una búsqueda, más probable es que su mejor score se deba en parte a la suerte. Para hacerlo visible, esta búsqueda usa solo 300 de los eventos sísmicos de entrenamiento de [3.5](3.5_multiclass_classification.ipynb) y además registra el score de prueba de cada candidato, algo que nunca usaríamos en una búsqueda real. Tarda unos 30 segundos.

In [12]:
SEISMIC_FILES = {
    "1000_earthquakes_physical_features.csv": "md5:28129c8dd1b3e14f655d489577b841b5",
    "1000_explosion_physical_features.csv": "md5:af1342d32e163e961e043364136359b0",
    "1000_noise_physical_features.csv": "md5:16cdb992fed6cf6273d5624f5df905da",
    "1000_surface_physical_features.csv": "md5:9a2c2643030cf058704d68e130654e9d",
}
try:
    tables = [pd.read_csv(pooch.retrieve(url=f"https://zenodo.org/api/records/14025693/files/{name}/content",
                                         known_hash=checksum, fname=name, path=pooch.os_cache("mlgeo")), index_col=0)
              for name, checksum in SEISMIC_FILES.items()]
    seismic = pd.concat(tables, ignore_index=True).dropna(axis=1)
    X = seismic.drop(columns=["source", "serial_no"])
    y = seismic["source"]
except Exception as error:
    print(f"Zenodo could not be reached ({error}). Using SYNTHETIC events: the scores below will differ from the text.")
    values, codes = make_classification(n_samples=4000, n_features=61, n_informative=20, n_classes=4, random_state=SEED)
    X = pd.DataFrame(values, columns=[f"feature_{i}" for i in range(61)])
    y = pd.Series(np.array(["earthquake", "explosion", "noise", "surface event"])[codes])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SPLIT_SEED, stratify=y)
truth = y_test.to_numpy()
print(f"{len(X_train)} training events, {len(X_test)} test events, {X.shape[1]} waveform features")

3000 training events, 1000 test events, 61 waveform features


In [13]:
macro = lambda a, b: f1_score(a, b, average="macro")
few = np.random.default_rng(1).choice(len(X_train), 300, replace=False)
X_few, y_few = X_train.iloc[few], y_train.iloc[few]
draw = np.random.default_rng(SEED)
tried = []
for _ in range(30):
    kind = draw.choice(["logistic", "forest", "nearest"])
    if kind == "logistic":
        model = make_pipeline(StandardScaler(), LogisticRegression(C=10 ** draw.uniform(-3, 2), max_iter=5000))
    elif kind == "forest":
        model = RandomForestClassifier(n_estimators=100, min_samples_leaf=int(draw.choice([1, 2, 5, 10, 20])),
                                       max_features=float(draw.choice([0.1, 0.33, 1.0])), random_state=SEED, n_jobs=-1)
    else:
        model = make_pipeline(StandardScaler(), KNeighborsClassifier(int(draw.choice([1, 3, 5, 10, 20, 40]))))
    tried.append({"kind": kind, "cv": cross_val_score(model, X_few, y_few, cv=5, scoring="f1_macro").mean(),
                  "test": macro(y_test, model.fit(X_few, y_few).predict(X_test))})
tried = pd.DataFrame(tried).sort_values("cv", ascending=False)
tried.head(5).round(3)

,kind,cv,test
21,forest,0.845,0.820
7,forest,0.826,0.809
12,forest,0.818,0.810
27,forest,0.818,0.810
8,forest,0.817,0.813


**Lea la tabla.** Compare el valor `cv` de la fila superior con el valor `test` de esa misma fila. La brecha es el optimismo de la búsqueda, y a menudo la fila superior no es la mejor en la prueba. Reporte el score de prueba del ganador, medido una sola vez, y nunca el mejor score de la búsqueda.

## Parte 3. Auditar un script escrito por un agente

Hoy, gran parte de la búsqueda la escribe un agente de programación en lugar de un paquete. Abajo está el tipo de script que produce un agente cuando se le pide «encontrar el mejor modelo para esta tabla sísmica». Se ejecuta e imprime un score alto.

In [14]:
# --- AI-generated modeling script: do NOT trust it yet ---
two_labels = seismic[seismic["source"] != "surface event"]   # remove noisy labels
features = StandardScaler().fit_transform(two_labels.drop(columns=["source", "serial_no"]))   # normalize features
a_train, a_test, b_train, b_test = train_test_split(features, two_labels["source"], test_size=0.25, random_state=7)

best_name, best_score = None, -1.0
for name, model in {"logistic_regression": LogisticRegression(max_iter=5000),
                    "random_forest": RandomForestClassifier(n_estimators=100, random_state=7, n_jobs=-1)}.items():
    model.fit(a_train, b_train)
    score = model.score(a_train, b_train)   # evaluate each candidate
    print(f"{name}: accuracy = {score:.3f}")
    if score > best_score:
        best_name, best_score = name, score
print(f"best model: {best_name}, accuracy = {best_score:.3f}")

logistic_regression: accuracy = 0.915
random_forest: accuracy = 1.000
best model: random_forest, accuracy = 1.000


```{admonition} Tarea: auditar el script
:class: attention
Tiene tres defectos distintos. Anótelos y, para cada uno, diga qué le hace al número reportado, antes de abrir la respuesta.
```

:::{dropdown} Los tres defectos
1. **Se elimina una clase.** `seismic["source"] != "surface event"` quita uno de los cuatro tipos de fuente bajo un comentario sobre etiquetas ruidosas. El score describe ahora un problema más fácil, y el modelo nunca puede predecir un evento superficial.
2. **El escalador se ajusta antes de la división.** `fit_transform` sobre todas las muestras usa los eventos de prueba para calcular medias y varianzas. Aquí el efecto es pequeño, pero con imputación o codificación del objetivo (*target encoding*) puede ser grande.
3. **Los modelos se eligen por su exactitud de entrenamiento.** `model.score(a_train, b_train)` premia la memorización: un bosque alcanza 1.000 sobre datos que ya ha visto.

Señales de alerta: un score casi perfecto, ningún modelo de referencia, ningún score sobre datos reservados.
:::

### Su turno (4): repárelo

Coloque el escalador dentro de un *pipeline* de scikit-learn (`Pipeline`) y use la división canónica con las cuatro clases. Calcule `fair_cv`, el macro-F1 del bosque con validación cruzada de 5 pliegues sobre `X_train, y_train`. Luego calcule `fair_test`, su macro-F1 sobre los eventos de prueba, una sola vez.

*Pista:* `pipeline = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=7, n_jobs=-1))`, luego `cross_val_score(pipeline, X_train, y_train, cv=5, scoring="f1_macro").mean()`, luego `macro(y_test, pipeline.fit(X_train, y_train).predict(X_test))`.

In [15]:
fair_cv = None     # replace None with your expression
fair_test = None   # replace None with your expression

*Solución desarrollada abajo (haga clic en ⋯ para abrirla).*

In [16]:
pipeline_sol = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=7, n_jobs=-1))
fair_cv_sol = cross_val_score(pipeline_sol, X_train, y_train, cv=5, scoring="f1_macro").mean()
fair_test_sol = macro(y_test, pipeline_sol.fit(X_train, y_train).predict(X_test))
check("fair_cv", globals(), fair_cv_sol, "use cross_val_score on X_train, y_train with scoring='f1_macro'.", tol=0.01)
check("fair_test", globals(), fair_test_sol, "fit the pipeline on all of X_train, then score on X_test.", tol=0.01)
print(f"Agent's script: accuracy 1.000 on three classes, on data it had seen")
print(f"Repaired: macro-F1 {fair_cv:.3f} in cross-validation, {fair_test:.3f} on test, four classes (always guessing one class: 0.10)")

fair_cv: not answered yet. Hint: use cross_val_score on X_train, y_train with scoring='f1_macro'.
  Using the reference result below.
fair_test: not answered yet. Hint: fit the pipeline on all of X_train, then score on X_test.
  Using the reference result below.
Agent's script: accuracy 1.000 on three classes, on data it had seen
Repaired: macro-F1 0.871 in cross-validation, 0.888 on test, four classes (always guessing one class: 0.10)


**Si tiene tiempo: pruébelo con su propio agente.** Pida a un agente de programación un AutoML corto que use solo scikit-learn, sobre los datos de su proyecto o sobre P395. Luego contraste su respuesta con esta lista: ¿qué deja fuera?, ¿qué modelo de referencia reporta?, ¿el escalador se ajusta solo sobre los pliegues de entrenamiento?, ¿el conjunto de prueba se usa una sola vez? Lleve lo que encuentre a la clase del viernes sobre evaluación de agentes ([6.3](../Chapter6-AgenticAI/6.3_build_an_eval_set.ipynb)).

## Qué conservar

La automatización pasó de los paquetes de búsqueda a agentes que escriben toda la cadena de modelado a pedido. Las verificaciones siguen siendo las mismas: un modelo de referencia, una división que corresponda a cómo se usará el modelo, validación cruzada dentro de los datos de entrenamiento y una sola mirada a un conjunto de prueba reservado. Automatice la búsqueda y verifique usted el resultado.